In [1]:
from transformers import BartTokenizer, BartForConditionalGeneration, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset
import pandas as pd
import torch
import os

import os
os.environ["WANDB_DISABLED"] = "true"

# ======================
# 1. Load raw HP text
# ======================
txt_path = "/kaggle/input/hp-all/merged_book.txt"
with open(txt_path, "r", encoding="utf-8") as f:
    raw_text = f.read().splitlines()

tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")

# Make dataset
dataset = Dataset.from_dict({"text": raw_text})
dataset = dataset.train_test_split(test_size=0.05, seed=42)

# Tokenize
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

# Data collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Model
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")

# Training args
training_args = TrainingArguments(
    output_dir="./bart_hp_pretrain",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=1,   # just adapt style, don’t overfit
    learning_rate=5e-5,
    save_strategy="epoch",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model("./bart_hp_pretrained")
tokenizer.save_pretrained("./bart_hp_pretrained")

2025-09-29 11:10:23.237567: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759144223.426153      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759144223.478413      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipykernel_19/3096805494.py:46: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
50,0.109900
100,0.037100
150,0.045300
200,0.017500
250,0.027800
300,0.012000
350,0.025600
400,0.022800
450,0.019700
500,0.029000


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


('./bart_hp_pretrained/tokenizer_config.json',
 './bart_hp_pretrained/special_tokens_map.json',
 './bart_hp_pretrained/vocab.json',
 './bart_hp_pretrained/merges.txt',
 './bart_hp_pretrained/added_tokens.json')

In [2]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

# ======================
# 1. Load CSV
# ======================
csv_path = "/kaggle/input/hp-all/hp_paraphrase_pairs_narrative_raw.csv"
df = pd.read_csv(csv_path)

dataset = Dataset.from_pandas(df)

def preprocess_function(examples):
    model_inputs = tokenizer(examples["input"], max_length=64, truncation=True, padding="max_length")
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["target"], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)

# ======================
# 2. Load pretrained HP-style BART
# ======================
model = BartForConditionalGeneration.from_pretrained("./bart_hp_pretrained")

# ======================
# 3. Training Args
# ======================
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_hp_finetuned",
    learning_rate=3e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=10,
    weight_decay=0.01,
    save_strategy="no",
    logging_steps=50,
    predict_with_generate=True,
    fp16=torch.cuda.is_available()
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# ======================
# 4. Trainer
# ======================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=None,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model("./bart_hp_finetuned")
tokenizer.save_pretrained("./bart_hp_finetuned")

Map:   0%|          | 0/64765 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3959: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipykernel_19/3189533911.py:46: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
50,8.107000
100,1.946700
150,0.571800
200,0.431200
250,0.425100
300,0.405300
350,0.400300
400,0.410800
450,0.383000
500,0.389900


('./bart_hp_finetuned/tokenizer_config.json',
 './bart_hp_finetuned/special_tokens_map.json',
 './bart_hp_finetuned/vocab.json',
 './bart_hp_finetuned/merges.txt',
 './bart_hp_finetuned/added_tokens.json')

In [3]:
from transformers import BartTokenizer, BartForConditionalGeneration

tokenizer = BartTokenizer.from_pretrained("./bart_hp_finetuned")
model = BartForConditionalGeneration.from_pretrained("./bart_hp_finetuned").cuda()

def bart_style_transfer(caption, max_new_tokens=60):
    inputs = tokenizer(caption, return_tensors="pt", truncation=True, padding=True, max_length=64).to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=5,
        repetition_penalty=1.2,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(bart_style_transfer("An old castle on a hill"))
print(bart_style_transfer("Students studying in a library"))

We need to wait until they’ve gone back to the castle.
“Library books are only as good as the teachers’ books.
